# 06 · Narrator and numeric grounding

**Role of this layer.** The narrator writes a short plain-English summary. It
never sees data series, only a **facts sheet** of numbers the executor computed,
already rounded (4 significant figures). It is told to use only those numbers,
exactly as displayed, and to do no arithmetic.

`check_grounding` then verifies that deterministically: every number in the
text must be a correct rounding of a computed fact (or its percentage form), a
plan parameter, a number from the request or a warning, an annualisation
factor, or a date within the data range; the 5% significance and 95% confidence
levels count only when a significance test ran. Without an API key a
deterministic template writes the narrative instead, and it passes the same check.

That check is guardrail **N01 UNGROUNDED_NARRATIVE** (action WARN, enforced by the
narrator check): an LLM narrative that fails it is withheld (kept in the run
record as `narrative.withheld`), the template narrative is shown instead, and an
N01 issue lists the numbers that failed. The numerical results are not affected.

In [1]:
import pandas as pd

from quantlab.config import load_settings

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)
settings = load_settings()

In [2]:
from quantlab.graph import run_graph
from quantlab.mock_scenarios import SCENARIOS_BY_ID
from quantlab.narrator import check_grounding

scenario = SCENARIOS_BY_ID["buy_the_dip_tech"]  # buy the dip in tech?
state = run_graph(scenario.request, list(scenario.dataset_ids), "mock", approve=True)
facts = state["narrative"].facts
pd.DataFrame([{"fact": f.fact_id, "value": f.value, "display": f.display} for f in facts]).head(20)

,fact,value,display
0,s1.n_obs,14042,"14,042"
1,s1.start,1971-02-05,1971-02-05
2,s1.end,2026-10-15,2026-10-15
3,s1.last_value,26713.03,"26,713"
4,s2.n_obs,2906,"2,906"
5,s2.start,1971-02-05,1971-02-05
6,s2.end,2026-10-09,2026-10-09
7,s2.last_value,27366.17,"27,366"
8,s3.n_obs,2905,"2,905"
9,s3.start,1971-02-12,1971-02-12


In [3]:
narrative = state["narrative"]
print(f"source: {narrative.source}")
print(narrative.text)
print("\ngrounding:", narrative.grounding)

source: template
The analysis used NASDAQCOM (1971-02-05 to 2026-10-15). 334 of 2,853 observations (a share of 0.1171) were flagged, in 258 distinct episodes. The test asks whether next-week returns after flagged weeks differ from those after the other weeks (two-sided): the difference is not statistically significant at the 5% level. The mean was higher after the other weeks: -0.001243 after flagged weeks against 0.002752 after the others. The significance test (s6) finds no statistically significant difference at the 5% level in 1-period forward returns between flagged and normal periods: the difference in means is -0.003995, with a Newey-West standard error of 0.002186 (8 lags), a p-value of 0.06756 and a 95% confidence interval from -0.008279 to 0.0002885. The 334 flagged observations (against 2,518 normal ones) form 258 distinct episodes, the real measure of independent evidence because flagged periods cluster; the test describes an association in this sample, not a cause or a for

The check catches invented numbers, and accepts correct roundings and
percentages. (`s3.last_value` is the latest weekly return in the facts sheet.)

In [4]:
latest = next(f for f in facts if f.fact_id == "s3.last_value").display
share = next(f for f in facts if f.fact_id == "s4.share_flagged").display
p_value = next(f for f in facts if f.fact_id == "s6.p_value").display
for sentence in [
    f"The latest weekly return is {latest}, and a share of {share} of the weeks was flagged.",
    f"The latest weekly return is about {float(latest) * 100:.1f}%.",
    f"No statistically significant difference at the 5% level (p-value {p_value}).",
    f"The p-value is {float(p_value) - 0.1:.4f}.",  # a wrong value
    "The Sharpe ratio of the strategy is 1.83.",
    "The NASDAQ peaked in 1955.",
]:
    result = check_grounding(sentence, facts, state["plan"], [scenario.request])
    print(f"{'OK ' if result.grounded else 'BAD'} {sentence}  ungrounded={result.ungrounded}")

OK  The latest weekly return is 0.006447, and a share of 0.1171 of the weeks was flagged.  ungrounded=[]
OK  The latest weekly return is about 0.6%.  ungrounded=[]
OK  No statistically significant difference at the 5% level (p-value 0.06756).  ungrounded=[]
BAD The p-value is -0.0324.  ungrounded=['0.0324']
OK  The Sharpe ratio of the strategy is 1.83.  ungrounded=[]
BAD The NASDAQ peaked in 1955.  ungrounded=['1955']


With an API key, the same facts go to the LLM narrator (live mode):

In [5]:
if settings.has_api_key:
    from quantlab.narrator import narrate
    from quantlab.nodes import warnings

    live = narrate(
        state["request"],
        state["plan"],
        state["artifacts"],
        warnings(state),
        "live",
        settings,
    )
    print(live.source, live.model, live.error)
    print(live.text)
    print("grounding:", live.grounding)
    print("N01 issues:", [issue.message for issue in live.issues])
else:
    print(
        f"No {settings.api_key_name} set: skipping the LLM narrator (the template is used above)."
    )

llm gpt-6-luna None
The two-sided test finds no statistically significant difference in the following week’s return between weeks flagged for bottom-10% NASDAQ returns and other weeks (p-value 0.06756). In this sample, the mean was higher for other weeks: 0.002752, compared with -0.001243 for flagged weeks. The difference in means (flagged minus other weeks) was -0.003995, based on 258 distinct flagged episodes. The expanding threshold used a minimum of 52 observations; the test covered 1972-02-11 to 2026-10-02.
grounding: grounded=True numbers_checked=9 ungrounded=[]
N01 issues: []
